In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage, AIMessage

# Load .env
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY not found in .env\n"
        "Add: GOOGLE_API_KEY=your_gemini_api_key"
    )

print("✅ Google Gemini API key loaded")

# Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.7,
)

# Store conversation history
messages = []

print("🤖 Gemini chatbot started!")
print("Type 'exit' to end the chat.\n")

while True:
    user_input = input("You: ")

    if user_input.strip().lower() == "exit":
        print("Chat ended.")
        break

    if not user_input.strip():
        continue

    try:
        # Add user's message
        messages.append(HumanMessage(content=user_input))

        # Send entire conversation to Gemini
        response = llm.invoke(messages)

        # Add Gemini response to memory
        messages.append(AIMessage(content=response.content))

        print("Bot:", response.content)

    except Exception as e:
        print("❌ Error:", e)


✅ Google Gemini API key loaded
🤖 Gemini chatbot started!
Type 'exit' to end the chat.



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [1]:
import psycopg
import os

conn = psycopg.connect(os.getenv("DATABASE_URL"))

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO memories
        (user_id, content, memory_type, importance)
        VALUES (%s, %s, %s, %s)
    """, (
        "user_1",
        "I am learning Generative AI and building a long-term memory system.",
        "fact",
        0.9
    ))

conn.commit()
conn.close()

print("Memory saved successfully!")

AttributeError: 'NoneType' object has no attribute 'encode'

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

DATABASE_URL = os.getenv("DATABASE_URL")

print("Database URL loaded:", DATABASE_URL is not None)

Database URL loaded: True


In [3]:
import psycopg
import os

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg.connect(DATABASE_URL)

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO memories
        (user_id, content, memory_type, importance)
        VALUES (%s, %s, %s, %s)
    """, (
        "user_1",
        "I am learning Generative AI and building a long-term memory system.",
        "fact",
        0.9
    ))

conn.commit()
conn.close()

print("Memory saved successfully!")

Memory saved successfully!


In [4]:
results = search_memory(
    "user_1",
    "What am I learning?"
)

for result in results:
    print(result)

NameError: name 'search_memory' is not defined

In [5]:
import psycopg
import os

def search_memory(user_id, query, limit=5):

    query_vector = embeddings.embed_query(query)

    conn = psycopg.connect(DATABASE_URL)

    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT
                content,
                memory_type,
                importance,
                1 - (embedding <=> %s::vector) AS similarity
            FROM memories
            WHERE user_id = %s
              AND embedding IS NOT NULL
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (
                str(query_vector),
                user_id,
                str(query_vector),
                limit
            )
        )

        results = cur.fetchall()

    conn.close()

    return results

In [6]:
results = search_memory(
    "user_1",
    "What am I learning?"
)

for result in results:
    print(result)

NameError: name 'embeddings' is not defined

In [7]:
results = search_memory(
    "user_1",
    "What am I learning?"
)

for result in results:
    print(result)

NameError: name 'embeddings' is not defined

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    output_dimensionality=768
)

print("✅ Embeddings ready")

✅ Embeddings ready


In [9]:
results = search_memory(
    "user_1",
    "What am I learning?"
)

for result in results:
    print(result)

('I am learning Generative AI and building a long-term memory system.', 'fact', 0.9, 0.6370393422524411)


In [10]:
results = search_memory(
    "user_1",
    "What am I learning?"
)

for result in results:
    print(result)

('I am learning Generative AI and building a long-term memory system.', 'fact', 0.9, 0.6370393422524411)


In [11]:
def get_memory_context(user_id, query):
    
    results = search_memory(
        user_id,
        query,
        limit=5
    )

    if not results:
        return "No relevant long-term memories found."

    memory_text = []

    for result in results:
        content = result[0]
        memory_text.append(f"- {content}")

    return "\n".join(memory_text)

In [12]:
def ask_with_memory(user_id, user_input):

    # 1. Search long-term memory
    memory_context = get_memory_context(
        user_id,
        user_input
    )

    # 2. Add retrieved memory to the question
    enhanced_input = f"""
Relevant long-term memories about the user:

{memory_context}

User's current question:
{user_input}

Use the relevant memories when answering the user.
Do not mention the memory system unless necessary.
"""

    # 3. Ask Gemini
    response = conversation.predict(
        input=enhanced_input
    )

    return response

In [13]:
response = ask_with_memory(
    "user_1",
    "What am I learning?"
)

print("Bot:", response)

NameError: name 'conversation' is not defined

In [14]:
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_classic.chains import ConversationChain

memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=4000
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

print("✅ Conversation ready")

c:\Users\Lenovo\Desktop\gen ai\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'llm' is not defined